In [ ]:
pip install transformers chromadb ultralytics

In [ ]:
import os
import json
import torch
import chromadb
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
from ultralytics import YOLO

INPUT_DIR = "/kaggle/input/datasets/parthanimesh/final-glance-dataset"  # B&W and multiple people sanitized dataset
DB_DIR = "/kaggle/working/chroma_db"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using {device}")

# scene encoder: full uncropped image, stock openai weights
scene_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
scene_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# garment encoder: fashion-tuned weights, runs on the YOLO person crop
fashion_model = CLIPModel.from_pretrained("patrickjohncyh/fashion-clip").to(device)
fashion_processor = CLIPProcessor.from_pretrained("patrickjohncyh/fashion-clip")

yolo_model = YOLO("yolov8n.pt")

client = chromadb.PersistentClient(path=DB_DIR)
collection = client.get_or_create_collection(
    name="glance_dataset",
    metadata={"hnsw:space": "cosine"},
)


def crop_and_square(img, box, pad_fraction=0.05):
    """Pad a YOLO box by 5%, crop it out, and square it with gray padding
    so the aspect ratio doesn't get distorted."""
    img_w, img_h = img.size
    x1, y1, x2, y2 = box

    pad_x = (x2 - x1) * pad_fraction
    pad_y = (y2 - y1) * pad_fraction
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(img_w, x2 + pad_x)
    y2 = min(img_h, y2 + pad_y)

    crop = img.crop((x1, y1, x2, y2))
    cw, ch = crop.size
    max_dim = max(cw, ch)

    square_img = Image.new("RGB", (max_dim, max_dim), color=(128, 128, 128))
    offset_x = (max_dim - cw) // 2
    offset_y = (max_dim - ch) // 2
    square_img.paste(crop, (offset_x, offset_y))
    return square_img


def get_clip_embedding(image, processor, model):
    # needs a dummy text input or the processor won't produce input_ids
    inputs = processor(images=image, text=[""], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        features = outputs.image_embeds
        features = features / features.norm(p=2, dim=-1, keepdim=True)
    return features.cpu().numpy()[0].tolist()


images_to_process = []
for root, _, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            images_to_process.append(os.path.join(root, file))

ids_batch = []
embeddings_batch = []
metadatas_batch = []

for filepath in tqdm(images_to_process, desc="Indexing"):
    file_name = os.path.basename(filepath)
    try:
        img = Image.open(filepath).convert("RGB")

        scene_vector = get_clip_embedding(img, scene_processor, scene_model)

        # dataset is pre-filtered to one prominent person per image
        results = yolo_model(img, verbose=False)
        boxes = results[0].boxes

        person_box = None
        for box in boxes:
            if int(box.cls[0].item()) == 0:  # class 0 = person
                person_box = box.xyxy[0].tolist()
                break

        if person_box is None:
            continue  # shouldn't happen on the clean dataset, but just in case

        squared_crop = crop_and_square(img, person_box)
        fashion_vector = get_clip_embedding(squared_crop, fashion_processor, fashion_model)

        ids_batch.append(file_name)
        embeddings_batch.append(fashion_vector)  # primary search key
        metadatas_batch.append({
            "filepath": filepath,
            "scene_vector_json": json.dumps(scene_vector),
            "person_bbox": json.dumps(person_box),
        })

        if len(ids_batch) >= 100:
            collection.add(ids=ids_batch, embeddings=embeddings_batch, metadatas=metadatas_batch)
            ids_batch, embeddings_batch, metadatas_batch = [], [], []

    except Exception as e:
        print(f"failed on {file_name}: {e}")

if ids_batch:
    collection.add(ids=ids_batch, embeddings=embeddings_batch, metadatas=metadatas_batch)

print("\ndone indexing")
print(f"total items: {collection.count()}")
print(f"db saved at: {DB_DIR}")

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/chroma_db_backup", 'zip', "/kaggle/working/chroma_db")

print("Database zipped successfully")